In [ ]:
!pip install fredapi yfinance --quiet
print("✓ Done")


✓ Done


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/CDS DS340/Project/data'
os.makedirs(DATA_DIR, exist_ok=True)
print(f'✓ Folder ready: {DATA_DIR}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Folder ready: /content/drive/MyDrive/CDS DS340/Project/data


In [ ]:
# CELL 3 — Download macro data from FRED
# ISM PMI: NAPM was discontinued on FRED.
# We use ISMMAN (ISM Manufacturing PMI) which is the correct current series.
# If that also fails we build a proxy from yield curve + consumer confidence.

import pandas as pd
import numpy as np
from fredapi import Fred
import time

FRED_API_KEY = '627e18837ffe7f9cda8422011db4123c'
fred = Fred(api_key=FRED_API_KEY)

# All series except ISM PMI (handled separately below)
FRED_SERIES = {
    'yield_curve':    'T10Y2Y',
    'consumer_conf':  'UMCSENT',
    'jobless_claims': 'IC4WSA',
    'lei':            'USSLIND',
    'gdp_real':       'GDPC1',
    'cpi':            'CPIAUCSL',
    'unemployment':   'UNRATE',
    'nber_recession': 'USREC',
}

def download_fred(series_id, name):
    try:
        raw = fred.get_series(
            series_id,
            observation_start='2000-01-01',
            observation_end='2024-12-31'
        ).dropna()
        gap = pd.Series(raw.index).diff().dropna().median()
        monthly = raw.resample('ME').last().ffill() if gap > pd.Timedelta('60 days') \
                  else raw.resample('ME').last()
        monthly.name = name
        print(f'  ✓ {name} ({series_id}): {len(monthly)} months')
        return monthly
    except Exception as e:
        print(f'  ✗ {name} ({series_id}): {e}')
        return pd.Series(name=name, dtype=float)

print('Downloading from FRED...')
series_list = []
for name, sid in FRED_SERIES.items():
    s = download_fred(sid, name)
    if not s.empty:
        series_list.append(s)
    time.sleep(0.2)

macro_df = pd.concat(series_list, axis=1).sort_index()
print(f'\nBase data shape: {macro_df.shape}')
print(f'Columns so far: {list(macro_df.columns)}')

  ✓ yield_curve (T10Y2Y): 300 months
  ✓ consumer_conf (UMCSENT): 300 months
  ✓ jobless_claims (IC4WSA): 300 months
  ✓ lei (USSLIND): 242 months
  ✓ gdp_real (GDPC1): 298 months
  ✓ cpi (CPIAUCSL): 300 months
  ✓ unemployment (UNRATE): 300 months
  ✓ nber_recession (USREC): 300 months

Base data shape: (300, 8)
Columns so far: ['yield_curve', 'consumer_conf', 'jobless_claims', 'lei', 'gdp_real', 'cpi', 'unemployment', 'nber_recession']


In [ ]:
# CELL 4 — Get ISM PMI (tries multiple series, builds proxy if all fail)

# ISM PMI series options on FRED (in order of preference)
PMI_OPTIONS = [
    ('ISMMAN',   'ISM Manufacturing PMI (current series)'),
    ('MANEMP',   'Manufacturing Employment (proxy)'),
    ('INDPRO',   'Industrial Production Index (proxy)'),
]

ism_data = None
print('Trying ISM PMI series...')

for sid, description in PMI_OPTIONS:
    try:
        raw = fred.get_series(
            sid,
            observation_start='2000-01-01',
            observation_end='2024-12-31'
        ).dropna()
        if len(raw) > 100:
            ism_data = raw.resample('ME').last()
            ism_data.name = 'ism_pmi'
            # Normalize to PMI-like scale (40-65) if not already
            if ism_data.mean() > 100 or ism_data.mean() < 30:
                ism_data = (ism_data - ism_data.mean()) / ism_data.std() * 5 + 52
            print(f'  ✓ Got ISM PMI from {sid} ({description}): {len(ism_data)} months')
            print(f'    Range: {ism_data.min():.1f} to {ism_data.max():.1f}')
            break
        else:
            print(f'  {sid}: only {len(raw)} obs, skipping')
    except Exception as e:
        print(f'  {sid} failed: {e}')
    time.sleep(0.2)

# If all FRED series failed, build a proxy
if ism_data is None:
    print('\nAll FRED PMI series unavailable — building proxy from yield curve + consumer confidence')
    print('(This is methodologically sound: both are leading indicators correlated with PMI)')
    yc     = macro_df['yield_curve']
    conf   = macro_df['consumer_conf']
    yc_n   = (yc   - yc.mean())   / yc.std()
    conf_n = (conf - conf.mean()) / conf.std()
    ism_data = (0.5 * yc_n + 0.5 * conf_n) * 5 + 52
    ism_data.name = 'ism_pmi'
    print(f'  ✓ Proxy built — range: {ism_data.min():.1f} to {ism_data.max():.1f}')

macro_df['ism_pmi'] = ism_data
print(f'\n✓ ism_pmi added to macro_df')

Trying ISM PMI series...
  ISMMAN failed: Bad Request.  The series does not exist.
  ✓ Got ISM PMI from MANEMP (Manufacturing Employment (proxy)): 300 months
    Range: 45.8 to 66.1

✓ ism_pmi added to macro_df


In [ ]:
# CELL 5 — Fix NaNs and save

print('Fixing NaN values...')

# LEI: gaps at start/end of series — forward/backward fill
macro_df['lei'] = macro_df['lei'].ffill().bfill()
print(f'  LEI NaNs remaining: {macro_df["lei"].isnull().sum()}')

# GDP: quarterly series, last 1-2 months of 2024 not yet reported — forward fill
macro_df['gdp_real'] = macro_df['gdp_real'].ffill()
print(f'  gdp_real NaNs remaining: {macro_df["gdp_real"].isnull().sum()}')

# Check everything
print(f'\nFinal NaN counts:')
print(macro_df.isnull().sum())

# Save to Drive
macro_df.to_csv(f'{DATA_DIR}/macro_raw.csv')

print(f'\n✓ Saved to {DATA_DIR}/macro_raw.csv')
print(f'  Shape: {macro_df.shape}')
print(f'  Columns: {list(macro_df.columns)}')
print(f'\nFinal check — last 3 rows:')
print(macro_df.tail(3))

Fixing NaN values...
  LEI NaNs remaining: 0
  gdp_real NaNs remaining: 0

Final NaN counts:
yield_curve       0
consumer_conf     0
jobless_claims    0
lei               0
gdp_real          0
cpi               0
unemployment      0
nber_recession    0
ism_pmi           0
dtype: int64

✓ Saved to /content/drive/MyDrive/CDS DS340/Project/data/macro_raw.csv
  Shape: (300, 9)
  Columns: ['yield_curve', 'consumer_conf', 'jobless_claims', 'lei', 'gdp_real', 'cpi', 'unemployment', 'nber_recession', 'ism_pmi']

Final check — last 3 rows:
            yield_curve  consumer_conf  jobless_claims   lei   gdp_real  \
2024-10-31         0.12           70.5        236250.0  1.72  23586.542   
2024-11-30         0.05           71.8        218750.0  1.72  23586.542   
2024-12-31         0.33           74.0        222750.0  1.72  23586.542   

                cpi  unemployment  nber_recession    ism_pmi  
2024-10-31  315.631           4.1             0.0  50.299616  
2024-11-30  316.528           4.2   

In [ ]:
# CELL 6 — Download sector ETF prices and returns from Yahoo Finance

import yfinance as yf

TICKERS = ['XLK','XLF','XLE','XLV','XLI','XLP','XLY','XLU','XLB','XLRE','XLC','SPY']

print('Downloading ETFs from Yahoo Finance...')
raw = yf.download(
    TICKERS,
    start='2000-01-01',
    end='2024-12-31',
    interval='1mo',
    auto_adjust=True,
    progress=True
)

prices = raw['Close']
prices.index = prices.index.to_period('M').to_timestamp('M')
prices.index.name = 'date'

returns = prices.pct_change().dropna(how='all')

prices.to_csv(f'{DATA_DIR}/sector_prices.csv')
returns.to_csv(f'{DATA_DIR}/sector_returns.csv')

print(f'\n✓ Prices saved:  {prices.shape}')
print(f'✓ Returns saved: {returns.shape}')
print(f'\nSample returns (last 3 months):')
print(returns.tail(3).round(4))

[*********************100%***********************]  12 of 12 completed



✓ Prices saved:  (300, 12)
✓ Returns saved: (299, 12)

Sample returns (last 3 months):
Ticker         SPY     XLB     XLC     XLE     XLF     XLI     XLK     XLP  \
date                                                                         
2024-10-31 -0.0059 -0.0267  0.0212  0.0173  0.0293 -0.0087 -0.0138 -0.0296   
2024-11-30  0.0596  0.0149  0.0691  0.0783  0.1046  0.0759  0.0517  0.0387   
2024-12-31 -0.0273 -0.1123 -0.0162 -0.1033 -0.0586 -0.0849 -0.0052 -0.0554   

Ticker        XLRE     XLU     XLV     XLY  
date                                        
2024-10-31 -0.0261 -0.0041 -0.0429 -0.0153  
2024-11-30  0.0417  0.0378  0.0037  0.1291  
2024-12-31 -0.0962 -0.0873 -0.0668  0.0092  
